<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/sessions/Session11_evaluating_and_publishing_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# جلسهٔ ۱۱: ارزیابی و انتشار مدل فاین‌تیون‌شده

### اهداف این جلسه

- ارزیابی دقیق مدل فاین‌تیون‌شده: دقت هر فیلد، تعمیم به الگوهای جدید و دادهٔ واقعی
- تحلیل خطا و تصمیم‌گیری دربارهٔ آمادگی مدل برای استفاده
- ادغام آداپتور LoRA با مدل پایه و نوشتن Model Card
- انتشار مدل در Hugging Face Hub
- تبدیل مدل به GGUF و اجرای آن با Ollama و API سازگار با OpenAI

> این نوت‌بوک به GPU نیاز دارد. در Colab از منوی `Runtime → Change runtime type` گزینهٔ **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۱. از آزمایش تا محصول

در جلسهٔ قبل مدلی آموزش دادیم که روی دادهٔ آزمون نتیجهٔ خوبی داشت. پیش از اینکه مدل در اختیار دیگران قرار بگیرد، به چند پرسش باید جواب داد:

1. **کجا اشتباه می‌کند؟** دقت کلی کافی نیست؛ باید بدانیم کدام فیلد و چه نوع پیامی مشکل دارد.
2. **روی دادهٔ واقعی چطور است؟** دادهٔ مصنوعی همیشه از دادهٔ واقعی تمیزتر است.
3. **دیگران چطور از آن استفاده کنند؟** مدل باید با مستندات، یعنی **Model Card**، و در قالبی قابل اجرا منتشر شود.

این نوت‌بوک آداپتور جلسهٔ ۱۰ را از پوشهٔ `order-extractor-lora` می‌خواند. اگر در Runtime جدیدی هستید و آداپتور را ندارید، سلول آموزش آن را دوباره می‌سازد.

</div>

In [ ]:
!pip -q install -U transformers trl peft bitsandbytes datasets huggingface_hub
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py

import json
import re

import pandas as pd
import torch
from llm_utils import download_data
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def load_jsonl(name):
    with open(download_data(name), encoding="utf-8") as f:
        return [json.loads(line) for line in f]


train_data, test_data, real_data = (load_jsonl(f"orders_{s}.jsonl") for s in ("train", "test", "real"))
test_split = ["seen templates"] * 150 + ["new templates"] * 50

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SYSTEM_SHORT = ("Extract the order from the Persian customer message as JSON with keys "
                "product_id, quantity, color, city.")
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
FIELDS = ["product_id", "quantity", "color", "city"]


def conversations(data):
    return [[{"role": "system", "content": SYSTEM_SHORT}, {"role": "user", "content": d["message"]}] for d in data]


def generate(model, convs, batch_size=16, max_new_tokens=80):
    tokenizer.padding_side = "left"
    outputs = []
    for i in range(0, len(convs), batch_size):
        texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in convs[i:i + batch_size]]
        batch = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        outputs += tokenizer.batch_decode(out[:, batch.input_ids.shape[1]:], skip_special_tokens=True)
    return outputs


def parse(output):
    match = re.search(r"\{.*\}", output, re.S)
    try:
        return json.loads(match.group()) if match else None
    except json.JSONDecodeError:
        return None


def same(a, b):
    norm = lambda x: None if x in (None, "", "null") else str(x).replace("\u200c", " ").strip()
    return norm(a) == norm(b)


def score(outputs, data):
    rows = []
    for out, item in zip(outputs, data):
        pred = parse(out) or {}
        row = {"message": item["message"], "output": out.strip(), "valid_json": bool(pred)}
        row.update({f: bool(pred) and same(pred.get(f), item["order"][f]) for f in FIELDS})
        row["all_fields"] = all(row[f] for f in FIELDS)
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
import os

ADAPTER_DIR = "order-extractor-lora"     # or a Hub id such as "your-username/order-extractor-lora"

if not os.path.isdir(ADAPTER_DIR) and "/" not in ADAPTER_DIR:
    # Same training as session 10 (about 5-10 minutes on a T4 GPU).
    from datasets import Dataset
    from peft import LoraConfig
    from trl import SFTConfig, SFTTrainer

    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto")
    dataset = Dataset.from_list([
        {"prompt": [{"role": "system", "content": SYSTEM_SHORT}, {"role": "user", "content": d["message"]}],
         "completion": [{"role": "assistant", "content": json.dumps(d["order"], ensure_ascii=False)}]}
        for d in train_data])
    tokenizer.padding_side = "right"
    trainer = SFTTrainer(
        model=model, train_dataset=dataset, processing_class=tokenizer,
        peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear",
                               task_type="CAUSAL_LM"),
        args=SFTConfig(output_dir=ADAPTER_DIR, num_train_epochs=2, per_device_train_batch_size=8,
                       gradient_accumulation_steps=2, learning_rate=2e-4, lr_scheduler_type="cosine",
                       warmup_steps=0.05, logging_steps=20, max_length=256, fp16=True, bf16=False,
                       save_strategy="no", report_to="none"))
    trainer.train()
    trainer.model.save_pretrained(ADAPTER_DIR)
    del model, trainer
    torch.cuda.empty_cache()
print("adapter:", ADAPTER_DIR)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۲. بارگذاری مدل پایه و آداپتور

</div>

In [ ]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()

test_scores = score(generate(model, conversations(test_data)), test_data).assign(split=test_split)
real_scores = score(generate(model, conversations(real_data)), real_data)
summary = pd.concat({
    "test (seen templates)": test_scores.query("split == 'seen templates'")[["valid_json", *FIELDS, "all_fields"]].mean(),
    "test (new templates)": test_scores.query("split == 'new templates'")[["valid_json", *FIELDS, "all_fields"]].mean(),
    "real messages": real_scores[["valid_json", *FIELDS, "all_fields"]].mean(),
}, axis=1).T.round(3)
summary

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۳. تحلیل خطا

جدول بالا نشان می‌دهد **کدام فیلد** بیشتر خطا دارد. حالا خود خطاها را می‌بینیم:

</div>

In [ ]:
pd.set_option("display.max_colwidth", 90)
errors = pd.concat([test_scores.query("not all_fields").assign(source="test"),
                    real_scores.query("not all_fields").assign(source="real")])
print(len(errors), "messages with at least one wrong field")
errors[["source", "message", "output"]].head(15)

In [ ]:
def error_type(row):
    wrong = [f for f in FIELDS if not row[f]]
    return "invalid JSON" if not row["valid_json"] else ", ".join(wrong)


errors.apply(error_type, axis=1).value_counts()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

الگوهای رایج خطا در این مسئله:

- **پیام‌های واقعی:** نوشتار محاوره‌ای که در دادهٔ مصنوعی نبود، مثل «واسه»، «برا»، «دوتا» بی‌فاصله یا «پوکو ایکس ۷» با حروف فارسی.
- **رنگ:** «نقره ای» با فاصله به جای نیم‌فاصله، یا رنگی که مشتری به شکل دیگری نوشته است.
- **کالا:** نام‌هایی که در داده نبوده‌اند. مدل ناچار به نزدیک‌ترین شناسه حدس می‌زند و هیچ‌وقت نمی‌گوید «نمی‌دانم».

مورد آخر مهم است: مدل فاین‌تیون‌شده **همیشه** یک شناسه برمی‌گرداند، حتی برای کالایی که در فروشگاه نیست. در محصول واقعی، خروجی مدل باید با فهرست کالاها اعتبارسنجی شود و پیش از ثبت سفارش، از مشتری تأیید گرفته شود. این را در جلسهٔ ۱۵ می‌سازیم.

### آیا مدل آماده است؟

این تصمیم فنی نیست، تصمیم کسب‌وکار است: دقت لازم را خطای قابل تحمل تعیین می‌کند. سیستمی که سفارش را **پیشنهاد** می‌دهد و مشتری تأیید می‌کند، خطای بیشتری را تحمل می‌کند تا سیستمی که سفارش را **مستقیم** ثبت می‌کند.

## ۴. ادغام آداپتور

برای انتشار و اجرا با ابزارهای دیگر، آداپتور را با وزن‌های مدل پایه **ادغام (Merge)** می‌کنیم: $W' = W + \frac{\alpha}{r}BA$ یک بار حساب و ذخیره می‌شود. ادغام باید روی مدل پایه با دقت **fp16** انجام شود، نه نسخهٔ ۴ بیتی، چون ادغام در وزن‌های کوانتیزه‌شده دقت را از دست می‌دهد.

</div>

In [ ]:
del model, base
torch.cuda.empty_cache()

base_fp16 = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16, device_map="auto")
merged = PeftModel.from_pretrained(base_fp16, ADAPTER_DIR).merge_and_unload()
merged.save_pretrained("order-extractor-merged")
tokenizer.save_pretrained("order-extractor-merged")
!du -sh order-extractor-merged

merged_scores = score(generate(merged, conversations(real_data)), real_data)
print("real messages, all fields correct:",
      f"4-bit + adapter {real_scores.all_fields.mean():.2f}   merged fp16 {merged_scores.all_fields.mean():.2f}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

نتیجهٔ مدل ادغام‌شدهٔ fp16 ممکن است کمی با نسخهٔ ۴ بیتی + آداپتور فرق کند، چون مدل پایه در دو دقت مختلف اجرا شده است. معمولاً نسخهٔ fp16 کمی بهتر یا برابر است.

## ۵. Model Card

هر مدل منتشرشده باید یک **Model Card** داشته باشد: فایل `README.md` که می‌گوید مدل چیست، با چه داده‌ای آموزش دیده، چقدر دقیق است، چطور استفاده می‌شود و **کجا نباید استفاده شود**. سربرگ YAML آن را Hugging Face برای جستجو و نمایش استفاده می‌کند.

مجوز مدل پایه را بررسی کنید: Qwen2.5-1.5B-Instruct با مجوز **Apache 2.0** منتشر شده و انتشار مدل مشتق‌شده از آن مجاز است. برخی مدل‌ها، مثل Llama، مجوزهای محدودتری دارند.

</div>

In [ ]:
s = summary
card = f"""---
license: apache-2.0
base_model: {BASE_MODEL}
language:
- fa
library_name: transformers
pipeline_tag: text-generation
tags:
- lora
- information-extraction
- persian
---

# Novin Kala order extractor

A {BASE_MODEL} model fine-tuned with QLoRA to turn informal Persian order messages of the fictional
Novin Kala online store into JSON with the keys `product_id`, `quantity`, `color` and `city`.
Built as an exercise in the [LLM Engineering Course](https://github.com/keyvanmahmoudi/LLM_Engineering_Course).

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "YOUR_USERNAME/order-extractor-qwen2.5-1.5b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
messages = [{{"role": "system", "content": "{SYSTEM_SHORT}"}},
            {{"role": "user", "content": "دو تا گلکسی A55 آبی برای شیراز می‌خوام"}}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
print(tokenizer.decode(model.generate(**inputs, max_new_tokens=80)[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
```

## Training data

1,200 synthetic messages generated from templates (`order_data.py` in the course repository).
Two epochs, LoRA r=16, alpha=32, all linear layers, 4-bit NF4 base model.

## Evaluation (all four fields correct)

| Test set | Accuracy |
|---|---|
| 150 messages, templates seen in training | {s.loc['test (seen templates)', 'all_fields']:.2f} |
| 50 messages, new templates | {s.loc['test (new templates)', 'all_fields']:.2f} |
| 12 hand-written real messages | {s.loc['real messages', 'all_fields']:.2f} |

## Limitations

- Knows only the 21 products of the fictional catalogue; for any other product it still returns one of these ids.
- Trained on synthetic data; accuracy on real customer messages is lower and was measured on 12 examples only.
- Output must be validated against the product catalogue and confirmed by the customer before an order is placed.
"""
with open("order-extractor-merged/README.md", "w", encoding="utf-8") as f:
    f.write(card)
print(card[:600])

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۶. انتشار در Hugging Face Hub

برای انتشار، یک **Access Token** با دسترسی نوشتن از [تنظیمات حساب Hugging Face](https://huggingface.co/settings/tokens) بسازید و با نام `HF_TOKEN` در بخش Secrets کولب ذخیره کنید. سلول زیر بدون توکن کاری انجام نمی‌دهد.

</div>

In [ ]:
from huggingface_hub import HfApi
from llm_utils import get_secret

hf_token = get_secret("HF_TOKEN")
if hf_token:
    api = HfApi(token=hf_token)
    repo_id = f"{api.whoami()['name']}/order-extractor-qwen2.5-1.5b"
    api.create_repo(repo_id, exist_ok=True)
    api.upload_folder(folder_path="order-extractor-merged", repo_id=repo_id,
                      commit_message="Upload order extractor")
    print("published:", f"https://huggingface.co/{repo_id}")
else:
    print("HF_TOKEN not set; skipping the upload")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

برای انتشار فقط آداپتور، که چند ده مگابایت است، همین کار را با پوشهٔ `order-extractor-lora` انجام دهید. کاربر آداپتور را با `PeftModel.from_pretrained` روی مدل پایه بارگذاری می‌کند.

## ۷. GGUF و اجرا با Ollama

**GGUF** قالب فایل پروژهٔ llama.cpp است و Ollama هم از آن استفاده می‌کند. مدل، توکن‌ساز و اطلاعات لازم در یک فایل قرار می‌گیرند و مدل می‌تواند با کوانتیزه‌سازی‌های مختلف ذخیره شود. اسکریپت `convert_hf_to_gguf.py` مدل Hugging Face را تبدیل می‌کند. نوع `q8_0` یعنی ۸ بیت، با کیفیتی تقریباً برابر fp16 و نصف حجم.

</div>

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip -q install -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python llama.cpp/convert_hf_to_gguf.py order-extractor-merged --outfile order-extractor-q8_0.gguf --outtype q8_0 2>&1 | tail -n 2
!ls -lh order-extractor-q8_0.gguf

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

در Ollama، مدل با یک **Modelfile** تعریف می‌شود: فایل GGUF، قالب گفتگو و پارامترهای پیش‌فرض. قالب گفتگو را صریحاً همان ChatML خانوادهٔ Qwen می‌گذاریم، تا مدل همان قالبی را ببیند که با آن آموزش دیده است.

</div>

In [ ]:
del merged, base_fp16
torch.cuda.empty_cache()

chat_template = ("{{- range .Messages }}<|im_start|>{{ .Role }}\n{{ .Content }}<|im_end|>\n"
                 "{{ end }}<|im_start|>assistant\n")
modelfile = (f'FROM ./order-extractor-q8_0.gguf\nTEMPLATE """{chat_template}"""\n'
             'PARAMETER stop <|im_end|>\nPARAMETER temperature 0\n')
with open("Modelfile", "w") as f:
    f.write(modelfile)
print(modelfile)

from llm_utils import setup_llm, start_ollama
start_ollama()
!ollama create order-extractor -f Modelfile
client, MODEL = setup_llm("ollama", model="order-extractor")

In [ ]:
ollama_outputs = []
for conv in conversations(real_data):
    response = client.chat.completions.create(model=MODEL, messages=conv, temperature=0)
    ollama_outputs.append(response.choices[0].message.content)

ollama_scores = score(ollama_outputs, real_data)
print(f"real messages, all fields correct (Ollama, q8_0): {ollama_scores.all_fields.mean():.2f}")
ollama_scores[["message", "output", "all_fields"]]

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

حالا مدل ما مثل هر مدل دیگری در Ollama با API سازگار با OpenAI در دسترس است. همهٔ کدهای جلسه‌های قبل، و برنامه‌ای که در ترم ۶ با FastAPI می‌سازیم، بدون تغییر می‌توانند از آن استفاده کنند. برای اجرا روی سیستم خودتان، فایل GGUF و Modelfile را دانلود کنید و همان دستور `ollama create` را اجرا کنید.

## ۸. جمع‌بندی

- ارزیابی پیش از انتشار: دقت هر فیلد، الگوهای جدید، دادهٔ واقعی و تحلیل نوع خطاها.
- مدل فاین‌تیون‌شده همیشه جواب می‌دهد، حتی وقتی نباید. خروجی را با دادهٔ معتبر بررسی کنید و از کاربر تأیید بگیرید.
- ادغام آداپتور را روی مدل fp16 انجام دهید، نه ۴ بیتی.
- Model Card کاربرد، داده، دقت، محدودیت‌ها و مجوز را مستند می‌کند. مجوز مدل پایه را پیش از انتشار بررسی کنید.
- GGUF و Ollama مدل را روی هر سیستمی و با API استاندارد قابل اجرا می‌کنند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## تمرین‌ها

هر تمرین را در سلول کد زیر آن حل کنید. پاسخ پیشنهادی در پوشهٔ `solutions` آمده است.

۱. مدل را با `--outtype f16` هم به GGUF تبدیل کنید. حجم دو فایل را مقایسه کنید و بررسی کنید روی چند درصد پیام‌های آزمون، خروجی دو مدل یکسان است.
۲. با استفاده از تحلیل خطای بخش ۳، بخش «Limitations» در Model Card را کامل‌تر کنید: دست‌کم سه نوع پیام را نام ببرید که مدل در آن‌ها خطا دارد، با یک مثال برای هر کدام.
۳. مدل Ollama را روی کل ۲۰۰ پیام آزمون ارزیابی کنید و دقت را با نتیجهٔ `transformers` در بخش ۲ مقایسه کنید. اگر تفاوت هست، علتش چه می‌تواند باشد؟
۴. دادهٔ آموزشی را هم به شکل یک **Dataset** در Hugging Face Hub منتشر کنید (`Dataset.push_to_hub`) و یک Dataset Card بنویسید که توضیح دهد داده مصنوعی است و چطور ساخته شده است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۱.** مدل را با `--outtype f16` هم به GGUF تبدیل کنید. حجم دو فایل را مقایسه کنید و بررسی کنید روی چند درصد پیام‌های آزمون، خروجی دو مدل یکسان است.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۲.** با استفاده از تحلیل خطای بخش ۳، بخش «Limitations» در Model Card را کامل‌تر کنید: دست‌کم سه نوع پیام را نام ببرید که مدل در آن‌ها خطا دارد، با یک مثال برای هر کدام.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۳.** مدل Ollama را روی کل ۲۰۰ پیام آزمون ارزیابی کنید و دقت را با نتیجهٔ `transformers` در بخش ۲ مقایسه کنید. اگر تفاوت هست، علتش چه می‌تواند باشد؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۴.** دادهٔ آموزشی را هم به شکل یک **Dataset** در Hugging Face Hub منتشر کنید (`Dataset.push_to_hub`) و یک Dataset Card بنویسید که توضیح دهد داده مصنوعی است و چطور ساخته شده است.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید